# LSTM · 07 The LSTM Model

Builds the LSTM architecture and counts its parameters. Nothing is trained here. **Saves:** nothing.

*Notebook 7 of 14 · Previous: 06 Class Weights · Next: 08 Training*

## Setup (the same in every notebook)

Each notebook in this project is self-contained. The cells below define the group settings and only the helper functions this notebook uses. They are identical in every notebook and nothing is imported from another file, so the notebook also runs on its own, for example in Colab.

In [1]:
# ---- Imports and dataset location ----------------------------------------------------------------
import sys, re, json, time, pathlib, textwrap
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from tensorflow import keras
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)

layers = keras.layers

# Locate dataset and split files locally
DATA_NAME = "deceptive-opinion.csv"
SPLIT_NAME = "split_assignment.csv"
here = pathlib.Path.cwd().resolve()
search_roots = [here, here.parent, here.parent.parent]

data_candidates = [r / "dataset" / DATA_NAME for r in search_roots] + [r / DATA_NAME for r in search_roots]
DATA_PATH = next((p for p in data_candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"Cannot find {DATA_NAME}. Ensure dataset/{DATA_NAME} exists.")

split_candidates = [r / "shared" / SPLIT_NAME for r in search_roots] + [r / SPLIT_NAME for r in search_roots]
SPLIT_PATH = next((p for p in split_candidates if p.exists()), None)
if SPLIT_PATH is None:
    raise FileNotFoundError(f"Cannot find {SPLIT_NAME}. Ensure shared/{SPLIT_NAME} exists.")

# Root folder for LSTM
LSTM_ROOT = next((p for p in [here, *here.parents] if (p / "01_Setup").is_dir() or (p / "LSTM_model.ipynb").exists()), here)
RESULTS = LSTM_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

print("Dataset:", DATA_PATH)
print("Split file:", SPLIT_PATH)
print("Results folder:", RESULTS)


Dataset: C:\USH_\The White Hat\DL Assignment\Deep-Learning---Assignment\dataset\deceptive-opinion.csv
Split file: C:\USH_\The White Hat\DL Assignment\Deep-Learning---Assignment\shared\split_assignment.csv
Results folder: C:\USH_\The White Hat\DL Assignment\Deep-Learning---Assignment\lstm\07_LSTM_Model\results


In [2]:
# ---- Group settings: identical for Simple RNN, LSTM, GRU and 1D CNN ------------------------------
SEED = 42

# Labels: this dataset has no star rating, so the `polarity` column is the label.
LABEL_COLUMN = "polarity"
LABEL_MAP = {"negative": 0, "positive": 1}

# Text -> integer sequences
VOCAB_SIZE = 5000          # total ids, including <PAD>=0 and <OOV>=1
MAX_LEN = 300              # tokens per review after cleaning
PAD_TOKEN, OOV_TOKEN = "<PAD>", "<OOV>"
PADDING = "pre"            # zeros in front, so a recurrent layer ends on real words
TRUNCATING = "post"        # long reviews keep their first MAX_LEN tokens

# Training
EMBED_DIM = 64
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MAX_EPOCHS = 30
PATIENCE = 5               # EarlyStopping patience, monitoring validation loss
MONITOR = "val_loss"
THRESHOLD = 0.5            # probability cut-off used for every model's test metrics

# Fixed 80/10/10 split loaded directly from shared/split_assignment.csv
split_df = pd.read_csv(SPLIT_PATH)
VAL_IDS = set(split_df.loc[split_df["split"] == "val", "row_id"])
TEST_IDS = set(split_df.loc[split_df["split"] == "test", "row_id"])

keras.utils.set_random_seed(SEED)          # seeds Python, NumPy and TensorFlow
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| seed", SEED)
print("Devices:", [d.device_type for d in tf.config.list_physical_devices()])
print(f"Loaded fixed split: {len(VAL_IDS)} validation rows, {len(TEST_IDS)} test rows.")


TensorFlow 2.21.0 | Keras 3.15.1 | seed 42
Devices: ['CPU']
Loaded fixed split: 160 validation rows, 160 test rows.


## The LSTM Model & Architecture Comparison (Unidirectional vs. Bidirectional)

```
Input (300 token ids)
  → Embedding(5000, 64)        each word id → a learned 64-number dense vector
  → LSTM(64)                   reads sequence token by token, returns final hidden state
  → Dropout(0.5)               regularization on recurrent features
  → Dense(32, ReLU)            non-linear combination of temporal features
  → Dropout(0.3)               regularization on dense features
  → Dense(1, Sigmoid)          probability that the review is positive P(positive)
```

### LSTM Theory & Gating Equations
A Long Short-Term Memory (LSTM) network resolves the vanishing and exploding gradient problems inherent in Simple RNNs through an explicit linear memory channel called the **Cell State** ($C_t$), controlled by three non-linear gating mechanisms:

1. **Forget Gate ($f_t$):** Controls what historical memory in $C_{t-1}$ to discard:
   $$f_t = \sigma(W_f x_t + U_f h_{t-1} + b_f)$$
2. **Input Gate ($i_t$) & Candidate Cell ($\tilde{C}_t$):** Determines what new information to write into the cell state:
   $$i_t = \sigma(W_i x_t + U_i h_{t-1} + b_i)$$
   $$\tilde{C}_t = \tanh(W_c x_t + U_c h_{t-1} + b_c)$$
3. **Cell State Update ($C_t$):** Preserves gradients across long sequences via addition:
   $$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$
4. **Output Gate ($o_t$) & Hidden State ($h_t$):** Filters which cell memory is exposed:
   $$o_t = \sigma(W_o x_t + U_o h_{t-1} + b_o)$$
   $$h_t = o_t \odot \tanh(C_t)$$

### Unidirectional vs. Bidirectional LSTM
* **Unidirectional LSTM:** Reads text forward from left to right ($t = 1 \to T$). It models context as it arrives sequentially, matching natural human reading.
* **Bidirectional LSTM:** Reads text in both forward ($t = 1 \to T$) and backward ($t = T \to 1$) directions using two separate LSTM layers. At each step, forward and backward hidden states are concatenated: $h_t = [\vec{h}_t, \overleftarrow{h}_t]$. This doubles recurrent parameters ($2 \times 33,024 = 66,048$) and produces a 128-dimensional output vector.

### Parameter Count Formula
For hidden units $H=64$ and input dimension $D=64$:
* Each gate has $(D + H) \times H + H = (64+64) \times 64 + 64 = 8,256$ parameters.
* An LSTM layer has 4 gates ($f, i, \tilde{C}, o$), giving $4 \times 8,256 = \mathbf{33,024}$ parameters (compared to 24,960 for GRU and 8,256 for Simple RNN).
* **Unidirectional LSTM Total:** Embedding (320,000) + LSTM (33,024) + Dense32 (2,080) + Dense1 (33) = **355,137** parameters.
* **Bidirectional LSTM Total:** Embedding (320,000) + Bi-LSTM (66,048) + Dense32 (4,128) + Dense1 (33) = **390,209** parameters.

--- 
### Hyperparameter Tuning Guide (Parameters to Modify)
To experiment with model performance and capacity, consider tuning:
* `output_dim` in `Embedding(VOCAB_SIZE, 64)`: Increase to 128 for richer representations or reduce to 32 to prevent overfitting.
* `units` in `LSTM(64)`: Adjust hidden representation capacity (e.g., 32, 64, 128).
* `Dropout`: Increase to 0.5 or 0.6 if training loss quickly drops to zero while validation loss rises (overfitting).
* `Bidirectional(...)`: Wrap in `Bidirectional` to provide future context, or use Unidirectional for faster, lighter execution.
* Layer Stacking: Add another layer with `layers.LSTM(64, return_sequences=True)` before the final LSTM layer.
* Classification Head: Modify intermediate `Dense(32, activation='relu')` units (e.g., 16, 32, 64).


In [3]:
# ==========================================================================================
# MODEL ARCHITECTURE DEFINITIONS WITH FINE-TUNING HOOKS
# ==========================================================================================

def build_unidirectional_lstm(units=64, embed_dim=EMBED_DIM, dropout1=0.5, dense_units=32, dropout2=0.3):
    """Unidirectional LSTM: processes sequence strictly forward (past to future)."""
    return keras.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(VOCAB_SIZE, embed_dim),                # <-- Modify 'output_dim' (e.g., 32, 64, 128)
        layers.LSTM(units, return_sequences=False),             # <-- Modify 'units' (e.g., 32, 64, 128)
        layers.Dropout(dropout1),                               # <-- Adjust dropout rate for regularization
        layers.Dense(dense_units, activation="relu"),           # <-- Modify intermediate Dense units (e.g., 16, 32, 64)
        layers.Dropout(dropout2),                               # <-- Adjust classifier dropout rate
        layers.Dense(1, activation="sigmoid"),                  # Binary output probability P(positive)
    ], name="unidirectional_lstm_sentiment")

def build_bidirectional_lstm(units=64, embed_dim=EMBED_DIM, dropout1=0.5, dense_units=32, dropout2=0.3):
    """Bidirectional LSTM: processes sequence in both forward and backward directions."""
    return keras.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(VOCAB_SIZE, embed_dim),                # <-- Modify 'output_dim' (e.g., 32, 64, 128)
        layers.Bidirectional(layers.LSTM(units, return_sequences=False)), # <-- Bidirectional LSTM wrapper
        layers.Dropout(dropout1),                               # <-- Adjust dropout rate for regularization
        layers.Dense(dense_units, activation="relu"),           # <-- Modify intermediate Dense units (receives 2*units)
        layers.Dropout(dropout2),                               # <-- Adjust classifier dropout rate
        layers.Dense(1, activation="sigmoid"),                  # Binary output probability P(positive)
    ], name="bidirectional_lstm_sentiment")

# Default build function for project standard interface
def build_lstm():
    return build_unidirectional_lstm()

def n_params(weights):
    return int(sum(np.prod(w.shape) for w in weights))

# Instantiate and inspect both architectures
uni_model = build_unidirectional_lstm()
bi_model = build_bidirectional_lstm()

print("=" * 70)
print("1. UNIDIRECTIONAL LSTM SUMMARY")
print("=" * 70)
uni_model.summary()

print("\n" + "=" * 70)
print("2. BIDIRECTIONAL LSTM SUMMARY")
print("=" * 70)
bi_model.summary()

uni_bd = {l.name: n_params(l.trainable_weights) for l in uni_model.layers if l.trainable_weights}
bi_bd = {l.name: n_params(l.trainable_weights) for l in bi_model.layers if l.trainable_weights}
print(f"\nUnidirectional Trainable params: {uni_bd} | Total: {uni_model.count_params():,}")
print(f"Bidirectional Trainable params: {bi_bd} | Total: {bi_model.count_params():,}")
print(f"Embedding share of parameters: {100 * next((v for k, v in uni_bd.items() if 'embedding' in k), 320000) / uni_model.count_params():.1f}%")


1. UNIDIRECTIONAL LSTM SUMMARY


Model: "unidirectional_lstm_sentiment"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 300, 64)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 355,137 (1.35 MB)

 Trainable params: 355,137 (1.35 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 300, 64)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 390,209 (1.49 MB)

 Trainable params: 390,209 (1.49 MB)

 Non-trainable params: 0 (0.00 B)


Unidirectional Trainable params: {'embedding': 320000, 'lstm': 33024, 'dense': 2080, 'dense_1': 33} | Total: 355,137
Bidirectional Trainable params: {'embedding_1': 320000, 'bidirectional': 66048, 'dense_2': 4128, 'dense_3': 33} | Total: 390,209
Embedding share of parameters: 90.1%


---
*Notebook 7 of 14 · Previous: 06 Class Weights · Next: 08 Training*